# Demo 11 - Cross-signal correlation

**Fast** (per-device aggregates) · **Pool:** Medium · **Visual:** clustered heatmap

**The question:** which of our telemetry signals actually tell us different things?

We build a table with one row per device and columns drawn from three separate sources -
processes, network and logons - then measure how strongly every pair of columns moves
together, and group the ones that behave alike.

Assembling a feature matrix across three tables and correlating every pair is heavy going
in KQL, and it has no way to draw the grouped result.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `WORKSPACE` - which workspace to read the three device tables from.
- `LOOKBACK_DAYS` - how much history to summarise per device.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 7

## 3. Build one row per device from three different telemetry sources

Three separate reads - processes, network and logons - each summarised per device, then
joined into a single table where every row is a device and every column is a behavioural
measure.

This is the part that is genuinely awkward in KQL: assembling a feature matrix from several
tables that do not share a schema. Here it is three group-bys and two joins.

Devices with no name are dropped. They join to nothing and would only add phantom rows.

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

def since(t):
    return (data_provider.read_table(t, WORKSPACE)
            .filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
            # a null DeviceName joins to nothing and adds phantom rows to the matrix
            .filter(F.col("DeviceName").isNotNull() & (F.col("DeviceName") != "")))

proc = since("DeviceProcessEvents").groupBy("DeviceName").agg(
        F.count("*").alias("proc_count"), F.countDistinct("FileName").alias("distinct_proc"))
net  = since("DeviceNetworkEvents").groupBy("DeviceName").agg(
        F.count("*").alias("net_count"), F.countDistinct("RemoteIP").alias("distinct_ips"))
logon= since("DeviceLogonEvents").groupBy("DeviceName").agg(
        F.count("*").alias("logon_count"), F.countDistinct("AccountName").alias("distinct_accts"))

feat = proc.join(net, "DeviceName", "outer").join(logon, "DeviceName", "outer").fillna(0).toPandas()
print("devices:", len(feat))

## 4. Correlate every signal against every other, and cluster the result

A **correlation** between two columns asks a simple question: when one goes up, does the
other? The answer is a number between -1 and +1.

- Close to +1: they move together (busy devices tend to be busy at everything)
- Close to 0: unrelated
- Close to -1: one rises as the other falls

A **clustermap** is that same correlation grid drawn as a heatmap, but with rows and
columns reordered so signals that behave alike sit next to each other, and a tree on the
edges showing how they group.

**What to look for:** blocks of signals moving together are telling you those measures are
partly redundant, which matters if you plan to feed them into a model. A pair you *expected*
to correlate but which does not is usually the more interesting result.

Constant columns are dropped first. A column where every device has the same value has no
correlation to compute, produces NaN, and the clustering step then fails outright - so the
cell falls back to a plain heatmap when that happens.

In [ ]:
cols = ["proc_count","distinct_proc","net_count","distinct_ips","logon_count","distinct_accts"]

# clustermap needs a finite matrix: a constant column produces NaN correlations and the
# linkage step then fails with "must contain only finite values".
varying = [c for c in cols if len(feat) > 1 and float(feat[c].std(ddof=0)) > 0]
corr = feat[varying].corr() if len(varying) >= 2 else None

if corr is None or corr.isna().to_numpy().any():
    print(f"Only {len(varying)} signal(s) vary across {len(feat)} device(s) - "
          "not enough to cluster. Raise LOOKBACK_DAYS.")
    if corr is not None:
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr.fillna(0), annot=True, cmap="coolwarm", center=0,
                    vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
        plt.title("Cross-signal correlation across device telemetry")
        plt.tight_layout(); plt.show()
else:
    g = sns.clustermap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1,
                       figsize=(8, 8), fmt=".2f", linewidths=.5)
    # .figure on seaborn >= 0.11.2, .fig on older releases
    getattr(g, "figure", getattr(g, "fig", None)).suptitle(
        "Cross-signal correlation across device telemetry", y=1.02)
    plt.show()

## Why a notebook beats KQL here

Assembling a per-entity feature matrix from three tables, correlating every pair, and clustering the result into a heatmap is a pandas/Seaborn one-liner. In KQL you'd hand-write each pairwise correlation and still have no clustered visual.